# 06 · CNC campaign — spatial EMFI sweep

(1) Jog + set the origin over the chip. (2) A `GlitchController` sweep with X/Y/Z as axes that moves the probe and fires the EMFI glitch at each point.

> ⚠️ HV: charges/fires the coil at every point. Shield on, nothing under the probe. STOP = `M112`; interrupt the kernel to stop firing.

## 1 · Manual control

Jog over the chip and **Set origin here** (`G92`) — that point is `(0,0,0)`. `permission denied`: `sudo chmod a+rw /dev/ttyUSB0`.

In [ ]:
import faultycat as fc

PORT, BAUD = "/dev/ttyUSB0", 115200   # e.g. /dev/ttyUSB0, /dev/ttyACM0, COM3
stage = fc.CncStage(PORT, BAUD)
stage

In [ ]:
fc.cnc_panel(stage)

### Work volume — separate interface

Define and **frame** the sweep region: set origin, capture corners or type bounds, **Frame volume** to trace it (returns to origin), and copy the `XS`/`YS`/`ZS` lines into the campaign.

In [ ]:
fc.cnc_volume_panel(stage)

## 2 · Connect the board and configure EMFI

`fc.connect()` opens the board; `immediate` trigger fires on command. Set pulse `delay`/`width` here. Version error → `faultycmd update`, or `fc.connect(allow_version_mismatch=True)` to try.

In [ ]:
cat = fc.connect()               # FaultyCat board (EMFI/crowbar/scanner)

cat.emfi.trigger = "immediate"   # fire on command, no external trigger
cat.emfi.delay_us = 0
cat.emfi.width_us = 5            # pulse width (start small)
cat

## 3 · Campaign: X / Y / Z axes + EMFI glitch per point

Axes are the CNC coordinates. `set_range` takes the values to visit (a list, or `range(0, 51)` for 0..50), not `(start, stop, step)`. Start small — each point is a move **and** an HV shot.

In [ ]:
XS, YS, ZS = [0.0, 5.0, 10.0], [0.0, 5.0, 10.0], [0.0]   # values to visit per axis
# (a list is the exact values, not start/stop/step; use range(0, 51) for 0..50)

gc = fc.GlitchController(["x", "y", "z"])
gc.set_range("x", XS)
gc.set_range("y", YS)
gc.set_range("z", ZS)

points = list(gc.glitch_values())
print(f"{len(points)} points, e.g. {points[0]} ... {points[-1]}")

Run: **move → fire** at each point. Iteration order = last axis fastest (with `z` fixed, `y` runs first per `x`).

In [ ]:
for i, p in enumerate(gc.glitch_values(), 1):
    fc.cnc_move_to(stage, p, feed=1500)                 # new position
    print(f"[{i}] at {p} -> charging + firing EMFI...")
    cat.emfi.glitch()                                   # charge (LED) -> fire (LED)

stage.goto_origin()   # back to the work origin (home) when finished
print("done")

Return the probe to the origin and release everything.

In [ ]:
stage.goto_origin()
stage.close()
cat.close()

## Next steps

Classify each shot from the target (`cat.uart`) and plot the spatial fault map — see nb 07.